### Цена, клик, группы и выдача

Четыре просьбы.

1. По площади, году и району угадать цену квартиры.
2. По признакам человека угадать, кликнет ли он по рекламе. На прошлых показах клик уже записан: да или нет.
3. Разложить транзакции на группы похожих. Подписей групп в таблице нет.
4. По запросу вернуть документы от самого полезного к менее полезному.

Как называется каждая задача и какой у неё ответ: число, метка, группа без имени или порядок? Почему четвёртую нельзя свести к регрессии «предскажи номер позиции 1, 2, 3, …»: что сломается, если в выдачу встанет новый документ полезнее текущего первого?


Ответ:
1. Задача на регрессию. Ответом будет число
2. Задача на классификацию. Ответом будет метка (0 - не кликнет, 1 - кликнет)
3. На кластеризацию. Ответом будет номер кластера
4. На ранжирование. Ответом будет упорядоченный список документов. Нельзя свести к регрессии, потому что если после предсказания появится новый документ полезнее всех остальных, нужно будет переобучать модель т.к. прежние значения уже не будут работать из-за сдвига



### Три вида признаков и масштаб

Квартира: число комнат, расстояние до метро в метрах, состояние «плохое < среднее < хорошее < отличное», название района.

Какой признак числовой, какой порядковый, какой категориальный? Почему район нельзя положить в модель числами 1, 2, 3 так, будто «центр» больше «севера»? Что делают с категорией, чтобы она стала столбцами из нулей и единиц, и почему один столбец из такой группы выкидывают, если рядом уже есть свободный член?

Ближайшие соседи ищутся по евклидову расстоянию. Расстояние до метро — тысячи, число комнат — единицы. Почему разница в 5 метров начинает весить как разница в 5 комнат? Какой простой ремонт это чинит и почему среднее и стандартное отклонение для него считают только по train?


Ответ:
Число комнат, расстояние - числовые, состояние квартиры - порядковый, название района - категориальный.
Районы нельзя положить в модель числами потому что могут возникнуть бессмысленные порядки и значения. Например, это будет означать, что район 3 в 3 раза сильнее района 1, или разница между районами 1 и 2 такая же, как и между 3 и 4. Для превращения категориальных признаков в числа используют векторизацию, чаще всего one-hot encoding. Если не удалить столбец, то в one-hot encoding сумма всех стобцов одной категории будет равна 1, а поэтому будет появляться линейная зависимость между признаками и свободным членом, то есть если взять функцию y = b + w1x1 + w2x2 + w3x3, где b - свободный член

			центр(x1)	север(x2)	юг(x3)
центр		1			0			0
север		0			1			0
юг			0			0			1
то, когда модель будет искать веса, получится 3 уравнения и 4 неизв, например:
Центр: b + w1 = 150
Север: b + w2 = 120
Юг: b + w3 = 110
Получается, что невозможно однозначно определить коэф.
Если же мы удаляем столбец, например, юг, то b становится базовым значением для юга и относительно него сравнивают все другие столбцы, получается 3 неизвестные и 3 уравнения, коэф определяются однозначно.
!!!!!!!!!!!!!!!!ДОПИСАТЬ!!!!!!!!!!

## 2. Как устроено обучение


### Прямая, парабола и степень 300

Один признак `x`. Семейство: `a(x) = θ0 + θ1 x + θ2 x² + … + θn xⁿ`. На картинке степень 1 проходит мимо облака, степень 2 ложится параболой, степень 300 проходит через учебные точки и между ними дико колеблется. У kNN при `k = 1` ошибка на обучающих точках тоже ноль: ближайший сосед учебной точки — она сама.

Какая из трёх степеней переобучена и по какому сравнению это видно, если кроме синих учебных точек есть зелёные, которых при подборе θ не было? Почему ноль на train совместим и с удачной простой формулой, и с зубрёжкой конечной выборки?


Ответ: Переобучена 3 модель, она очень хорошо подстроилась под учебные данные и прошла через много синих точек, но не научилась правильно находить закономерность, т.к. слишком сильно колеблется. Это видно по сравнению ошибок на тренировочных и тестовых данных. На тренировочных ошибка будет почти нулевая, а на тестовой намного выше. Ноль на train совместим и с удачной простой формулой, и с зубрежкой конечной выборки т.к. нулевая ошибка на трейн говорит только о том, что модель идеально предсказала уже известные ей данные, а она могла либо найти закономерность, либо, если произошло переобучение зазубрить трейн.

### Train, validation, test

Выборку разделили на три части. Веса θ подбирают на первой. На второй после каждого круга решают, менять ли семейство и гиперпараметры: перебрали `k` от 10 до 5 и оставили 5, потому что там ошибка меньше. Третью часть смотрят один раз в конце.

Как называется каждая часть? Почему выбранное `k = 5` модель фактически узнала со второй части, хотя веса на ней не учили? Зачем тогда третья, если вторая уже ловит переобучение?


Ответ: Первая часть - train, вторая - validation, третья - test. k = 5 мы подобрали, потому что модель выдала наименьшую ошибку при таком гиперпараметре. 3 часть test необходима для независимой итоговой оценки на данных, которые она еще не видела.

### Лосс и метрика

Лосс — число от истинного ответа и предсказания: чем больше, тем хуже. Эмпирический риск — средний лосс по обучающей выборке. Обучение — выбрать θ, на котором этот средний лосс меньше.

Метрика тоже сравнивает предсказания с ответами, но веса под неё не подбирают.

Почему в регрессии MSE часто бывает и лоссом, и метрикой, а в классификации так почти никогда: лосс считают от вероятности, accuracy — уже от выбранной метки? Кому из двоих нужно, чтобы по числу было удобно искать минимум, а кому — чтобы число можно было объяснить заказчику?


Ответ: MSE в регрессии часто бывает и лоссом, и метрикой, т.к. одновременно подходит для оптимизацици и оценки. В классификации используют log loss, т.к. этот показатель честнее штрафует модель за неуверенный ответ. Лоссу - чтобы удобно искать минимум, метрике - чтобы объяснить заказчику.

## 3. Среднее, разброс, связь


### Типичная зарплата

Суммы уже посчитаны.

| Набор | Числа | Среднее | Медиана | Мода |
| --- | --- | --- | --- | --- |
| Зарплаты, тыс. | 40, 42, 45, 45, 48, 50, 52, 90 | 51.5 | 46.5 | 45 |
| Оценки | 3, 4, 4, 5, 4 | 4 | 4 | 4 |

Нужна одна цифра «типичная зарплата» для обычного сотрудника. Какую называешь и почему 51.5 описывает не его? В каком ряду среднее, медиана и мода не спорят?

Константный прогноз «всем одну цену» оптимален в разных точках. Для MSE эта точка — среднее, для MAE — медиана. Как это видно на ряде зарплат: куда 90 утаскивает среднее и где остаётся медиана?


Ответ: Назову медиану - 46.5. 51.5 не описывает правильно типичную зарплату, т.к. в выборке наблюдается выброс - 90, из-за него только у 2 человек из 8 зарплата больше заявленной средней. MSE возводит в квадрат, поэтому она сильно штрафует модель за большую ошибку в выбросе. 90 вытягивает среднее вправо. MAE учитывает линейно, поэтому если выбросов немного, то медиана остается устойчивой

### Свойства среднего и дисперсии

Дисперсия здесь — сумма квадратов отклонений от своего среднего, делённая на n. Стандартное отклонение — корень из неё.

1. К каждой зарплате прибавили 10 тысяч премии. Куда сдвинется среднее? Почему дисперсия не изменится?
2. Каждую зарплату умножили на 2. Что станет со средним? Почему дисперсия вырастет в 4 раза, а стандартное отклонение только в 2?


Ответ: 1. Среднее сдвинется вправо на 10 тысяч, дисперсия не изменится, т.к. вместе с зарплатами сдвинулось и среднее
2. Дисперсия вырастет в 4 раза, т.к. дисперсия квадратично зависит от масштаба данных, если любое число выборки умножить на a, то дисперсия увеличится в a^2 раз. Стандартное отклонение - корень из дисперсии. Корень из 4 - 2

### Размах, квартили и ящик

Размах — разница максимума и минимума. На зарплатах он большой из-за одного значения 90.

Почему одного размаха мало как меры разброса всего ряда? Что на ящике с усами означают нижняя граница ящика, черта внутри, верхняя граница и отдельные точки за усами? Какая доля наблюдений сидит внутри ящика? Почему по ящику видно и середину, и разброс, и далёкую точку, а по одному среднему — нет?


Размаха мало потому что он учитывает только 2 крайних значения и не показывает как распеределены остальные. Нижняя граница ящика - первый квартиль, черта внутри - 2, верхняя - 3, точки за усами - выбросы. Внутри ящика находятся центральные 50% выборки. Ящик учитывает больше параметров чем среднее, среднее сильнее реагирует на выбросы.

### Корреляция

В декабре вместе выросли и расходы на рекламу, и продажи. В отчёте: «реклама подняла продажи, ряды движутся вместе».

Какую фразу можно сказать честно? Какая третья величина не отрезана?

Пирсон лежит от −1 до 1 и ловит только линейную связь. Одна очень далёкая точка может сильно сдвинуть его. Что можно сказать, если коэффициент около нуля, а облако на рассеянии — дуга, а не каша? Почему из нуля ещё не следует, что величины вообще никак не связаны?


Ответ: В декабре наблюдается положительная корреляция между расходами на рекламу и продажами. Если Пирсон равен нулю, то это еще не значит что зависимости нет, она может быть просто нелинейной. Например, если облако точек образует дугу, то зависимость y = x**2.

## 4. Линейные модели


### Вес, сдвиг и новые колонки

Модель `a(x) = w0 + w1 x1 + … + wn xn`. Предсказание — одно скалярное произведение, если к признакам дописать единицу и спрятать `w0` в общий вектор весов.

Что говорит знак веса при признаке и что говорит большой модуль веса? При каком условии эта расшифровка врёт: признак в других единицах, чем соседи, или две колонки почти повторяют друг друга?

Почему модель остаётся линейной по весам, если вместо `x` в колонку положить `x²` или логарифм `x`? Где тогда живёт нелинейность — в весах или в сборке признаков?


Вес определяет как изменится предсказание модели, если знак положительный - предсказание увеличивается, минус - уменьшается. Расшифровка врет когда у признаков разные масштабы. Модель остается линейной, потому что она линейна относительно весов, признаки уже известны и заморожены, а веса - неизвестные переменные.

### Шаг против градиента

У квадратичного лосса минимум можно выписать формулой `(XᵀX)⁻¹ Xᵀ y`, если столбцы независимы. У LogLoss такой формулы нет. Минимум ищут шагами: посчитать градиент и вычесть его, умноженный на длину шага α.

Куда смотрит градиент и почему шаг идёт в обратную сторону? Что будет с весами за одно и то же число итераций, если α очень маленький и если α слишком большой? Почему α не учат вместе с весами?

Полный градиент смотрит на все строки. Шаг по случайным k строкам называют батчем. Почему кривая лосса при `k = 1` дёргается сильнее, чем при `k = 16`, даже если оба варианта в конце приходят примерно на один уровень?


Ответ: Градиент показывает направление в котором функция быстрее всего возрастает, шаг идет в обратную сторону потому что нам нужно уменьшать ошибку а не увеличивать. Очень маленький α - маленькие шаги, за одинаковое число итераций можно не успеть приблизиться к минимуму, слишком большой α - риск перепрыгнуть через минимум или удалиться от него. α не учат вместе с весами, потому что алгоритм может начать изменять размер шага, вместо того чтобы искать подходящие веса.
Кривая лосса дергается сильнее потому что при k=1 градиент сильно реагирует на шум и обновляет веса на каждом объекте, а при k=16 вклад отдельных объектов компенсируется поэтому получается более стабильно

### L1, L2 и шкала

Если столбцы линейно зависимы, одни и те же предсказания дают разные веса, и среди них есть сколь угодно большие. К лоссу дописывают штраф. Свободный член в штраф не кладут.

Чем L2 (Ridge) отличается от L1 (Lasso) по тому, что происходит с весами? У кого из них снова есть закрытая формула, а кто зануляет часть признаков? Почему коэффициент штрафа подбирают по validation, не по test?

Зачем перед штрафом привести признаки к одному масштабу: вычесть среднее train и поделить на стандартное отклонение train? Что сломается в чтении «большой вес = важный признак», если один столбец в метрах, а другой в штуках?


Ответ: Ridge (L2) уменьшает веса, стягивая их к нулю, но обычно не обнуляет полностью, Lasso (L1) может полностью обнулять некоторые веса, тем самым выполняя отбор признаков. Ridge имеет закрытую формулу, а Lasso зануляет. 
Коэф подбирают на validation, чтобы не допустить переобучения на тестовых данных. Validation используется для подбора гиперпараметров, а Test - для независимой финальной оценки качества модели.
Приводят к одному масштабу, чтобы регуляризация справедливо штрафовала веса разных признаков. Вес не будет показывать важность, если один столбец в метрах, а другой в штуках, т.к. единицы измерения абсолютно разные, например количество комнат и расстояние до метро.

### Логистическая регрессия

Линейная часть та же, `t = w0 + w·x`. Сигмоида `1 / (1 + e^(−t))` сжимает t в интервал (0, 1). Это предсказанная вероятность класса 1, ещё не метка.

Почему метод называется регрессией, хотя задача — клик или не клик? Почему порог 0.5, которым вероятность превращают в метку, не является частью обучения и не обязан быть лучшим?

Квадрат ошибки не знает, что ответ должен быть вероятностью. Лосс одного объекта — минус логарифм вероятности его истинного класса. Почему уверенная ошибка (вероятность 0.99 при верном классе 0) бьёт по этому лоссу сильнее, чем осторожная (вероятность 0.6 при том же классе)?


Ответ: Это регрессия, потому что модель вычисляет непрерывное значение, а сигмоида преобразует его в вероятность класса 1.
Порог применяется после обучения для превращения вероятности в метку. Его можно менять в зависимости от задачи и цены ошибок.
Потому что logloss сильно штрафует за уверенные неправильные предсказания.

## 5. Метрики регрессии


### MSE, MAE, RMSE, MAPE и R²

Спрос был 15. Предсказали 20 — лишний склад. Предсказали 10 — товара не хватило, и это хуже для магазина. MSE и MAE штрафуют отклонение вверх и вниз одинаково.

Почему для этой просьбы одной MSE мало, даже если она удобна как лосс? Чем RMSE удобнее MSE, когда ошибку надо назвать в тех же единицах, что и спрос? Почему MAPE нельзя посчитать, если верный ответ нулевой?

R² сравнивает модель с константой «всегда предсказывать среднее ответов». Что означает R² = 0, R² = 1 и отрицательный R²?


Ответ: MSE одинаково штрафует завышение и занижение прогноза, но в бизнесе нехватка товара обычно хуже чем избыток , поэтому нужны еще метрики.
RMSE выражается в тех же единицах измерения, что и исходная величина. MSE выражается в квадратных единицах
Потому что MAPE делит ошибку на истинное значение. При y = 0 происходит деление на ноль.
R**2 = 1 идеальные предсказания.
R**2 = 0 модель не лучше предсказания средним значением.
R**2 < 0 модель хуже, чем простое предсказание среднего.

## 6. Какой график зачем

Для каждой просьбы назови один график и что на нём смотреть.

1. Как распределены зарплаты: где густо и есть ли хвост в сторону больших значений.
2. Как зарплата отличается по отделу и нет ли точек далеко от ящика.
3. Растёт ли стаж вместе с зарплатой или облако рассыпано, в том числе дугой, которую Пирсон не поймает.
4. Много числовых столбцов сразу: где линейная связь сильная. Почему на диагонали тепловой карты стоят единицы и на них не смотрят.
5. Как число клиентов менялось от месяца к месяцу.


Ответ: 1. Гистограмма. Показывает, где сосредоточено больше всего зарплат
2. Ящик с усами. Показывает медиану, разброс зарплат и выбросы

## 7. Git


### Своя ветка, commit и push

Сдача — этот ноутбук в папке `practice`. Основная ветка репозитория уже есть, файл сразу в неё не кладут.

Зачем отдельная ветка от основной и что в ней должно оказаться к сдаче? Чем `git commit` отличается от `git push`, если файл уже сохранён на диске? Если push пишет, что на сервере есть коммит, которого у тебя нет, почему сначала забирают историю, а отправка, которая затирает чужое на сервере, здесь не починка?


Ответ: Отдельная ветка нужна, чтобы работать независимо от основной. В отдельной ветке должнен находиться выполненный ноутбук в папке practice. 
git commit сохраняет изменения в локальной истории git
git push отправляет локальные коммиты на удаленный сервер github
Сохранение файла на диске не создает коммит
Сначала забирают историю потому что на сервере могут быть изменения которых нет локально. Принудительный пуш может затереть чужие коммиты.	

## 8. Задачи на Python

Свои функции, списки и словари. sklearn и готовые среднее, MSE, сигмоида не используются.

Договорённости:

- среднее — сумма, делённая на количество;
- при чётном количестве медиана — среднее двух центральных после сортировки;
- мода — значение, которое встречается строго чаще остальных; если такого нет, верни `None`;
- дисперсия — сумма квадратов отклонений от среднего, делённая на n;
- стандартное отклонение — корень из этой дисперсии.


### Задача 1. Среднее, медиана, мода

Напиши `mean`, `median` и `mode`. Выведи три числа для каждого ряда.

- ряд A: `40, 42, 45, 45, 48, 50, 52, 90`
- ряд B: `1, 2, 3, 4`


In [10]:
a = [40, 42, 45, 45, 48, 50, 52, 90]
a.sort()
def mode_find(lst):
	freq = {}
	for elem in lst:
		freq[elem] = freq.get(elem, 0) + 1
	if len(freq) > 1 and len(set(freq.values())) == 1:
		return None
	return max(freq, key=freq.get)

mean = sum(a)/len(a)
median = (a[len(a)//2] + a[len(a)//2-1])/2
mode = mode_find(a)
print(mean, median, mode)

b = [1, 2, 3, 4]
b.sort()
mean = sum(b)/len(b)
median = (b[len(b)//2] + b[len(b)//2-1])/2
mode = mode_find(b)
print(mean, median, mode)

51.5 46.5 45
2.5 2.5 None


### Задача 2. Размах, дисперсия, стандартное отклонение

Ряд: `1, 2, 3, 4, 5`.

Напиши `data_range`, `variance` и `std`. Выведи три числа для исходного ряда, для ряда «к каждому элементу прибавить 10» и для ряда «каждый элемент умножить на 2».

По выводу должно быть видно: сдвиг дисперсию не меняет, удвоение умножает дисперсию на 4, а стандартное отклонение на 2.


In [15]:
a = [1, 2, 3, 4, 5]
b = [x+10 for x in a]
c = [x*2 for x in a]

def variance(a):
	mean = sum(a)/len(a)
	d = 0
	for x in a:
		d += (mean-x)**2
	d /= len(a)
	return d


def data_range(a):
	return max(a) - min(a)

def std(a):
	return variance(a)**0.5
print(data_range(a), variance(a), std(a))
print(data_range(b), variance(b), std(b))
print(data_range(c), variance(c), std(c))



4 2.0 1.4142135623730951
4 2.0 1.4142135623730951
8 8.0 2.8284271247461903


### Задача 3. MSE, MAE, RMSE и R²

Верные ответы `y = [2, 4, 6, 8]`, предсказания `pred = [3, 3, 7, 9]`.

Напиши функции и выведи четыре числа. RMSE оставь корнем, не округляй заранее; R² можно оставить дробью, которую посчитает деление.

- MSE — среднее квадратов `(pred − y)`;
- MAE — среднее модулей `(pred − y)`;
- RMSE — корень из MSE;
- R² = `1 − (сумма квадратов (y − pred)) / (сумма квадратов (y − среднее y))`.


In [ ]:
import math
y = [2, 4, 6, 8]
pred = [3, 3, 7, 9]
def mse(y, pred):
    return sum((p - a) ** 2 for a, p in zip(y, pred)) / len(y)

def mae(y, pred):
    return sum(abs(p - a) for a, p in zip(y, pred)) / len(y)

def rmse(y, pred):
    return math.sqrt(mse(y, pred))

def r2(y, pred):
    mean_y = sum(y) / len(y)
    ss_res = sum((a - p) ** 2 for a, p in zip(y, pred))
    ss_tot = sum((a - mean_y) ** 2 for a in y)
    return 1 - ss_res / ss_tot

print(mse(y, pred))
print(mae(y, pred))
print(rmse(y, pred))
print(r2(y, pred))

### Задача 4. Один шаг градиентного спуска

Один признак, модель `pred = w0 + w1 * x`. Старт: `w0 = 0`, `w1 = 1`. Точки: `x = [1, 3]`, `y = [2, 4]`. Длина шага `lr = 0.1`.

Сначала посчитай предсказания. Потом градиент среднего квадрата ошибки, как на семинаре. Здесь `resid = pred − y`, n — число точек:

- `grad_w0 = (2 / n) * сумма resid`
- `grad_w1 = (2 / n) * сумма (resid * x)`

Один шаг: `w = w − lr * grad`. Выведи новые `w0` и `w1`.

Свободный член в этой задаче есть, штрафа на веса нет: это ещё не Ridge и не Lasso, только шаг по MSE.


In [ ]:
w0 = 0
w1 = 1
x =  [1, 3]
y = [2, 4]
n = len(x)
lr = 0.1
def pred(w0, w1, x):
	return w0 + w1*x

prediction = [pred(w0, w1, x1) for x1 in x]
resid = [p - yi for p, yi in zip(x, y)]
grad_w0 = 2/n * sum(resid)
grad_w1 = 2/n * sum(r*xi for r, xi in zip(resid, x))
w0 = w0 - lr*grad_w0
w1 = w1 - lr*grad_w1
print(w0)
print(w1)

0.2
1.4


### Задача 5. Сигмоида и LogLoss

Сигмоида: `sigmoid(t) = 1 / (1 + e^(−t))`. Лосс одного объекта при `y` равном 0 или 1 и вероятности `p`:

`L = −y * log(p) − (1 − y) * log(1 − p)`

Напиши обе функции.

1. Выведи `sigmoid(0)` и `sigmoid(2)`, второе округли до трёх знаков через `round(..., 3)`.
2. Выведи лосс для пары `(y = 1, p = 0.8)` и для пары `(y = 0, p = 0.8)`, тоже до трёх знаков.
3. По порогу 0.5 преврати вероятности `[0.2, 0.8]` в метки: если `p >= 0.5`, класс 1, иначе 0. Выведи список меток.


In [23]:
import math
def sigmoid(t):
	return 1 / (1 + math.exp(-t))

def logloss(y, p):
	return -y * math.log(p) - (1 - y) * math.log(1 - p)

print(sigmoid(0), round(sigmoid(2), 3))
print(round(logloss(1, 0.8), 3), round(logloss(0, 0.8), 3))
probs = [0.2, 0.8]
labels = [1 if x >= 0.5 else 0 for x in probs]
print(labels)


0.5 0.881
0.223 1.609
[0, 1]
